In [ ]:
from crewai_tools import RagTool
from crewai import Agent, Task, Crew, Process

#config
config = {
    "vectordb": {                               # 문서의 벡터를 저장하고 검색하는 역할을 담당하는 벡터 데이터베이스를 정의함.
        "provider": "chromadb",                 # 로컬 환경에서도 가볍고 빠르게 사용가능
        "config": {
            "collection_name": "kb-collection"  # 벡터를 저장할 논리적 공간의 이름. 반드시 명시해야 하는 필수값.
        }
    },

    "embedding_model": {                        # 텍스트를 벡터 형태로 변환하는 임베딩 모델을 지정함.
        "provider": "openai",
        "config": {
            "model_name": "text-embedding-3-small"
        }
    },

    "chunker": {
        "chunk_size": 300,                      # 긴 문서를 어떤 단위로 나누어 벡터화 할지 결정함
        "chunk_overlap": 50,
        "length_function": "len",               # 청크 길이를 문자수 기준으로 계산하도록 "len"으로 사용.
        "min_chunk_size": 0
    }
}

In [ ]:
# 이제 이렇게 정의된 config 를 실제 RAG 도구에 적용하고, 해당 도구를 사용하는 에이전트와 크루를 구성해 보겠음.

# 아래 코드는 KB 주택 시장 리류 PDF 연구 보고서 사이트를 기반으로 "질문-> RAG 검색 -> 분석 리포트 작성"까지 한번에 수행하는 간단하 멀티 에이전트 예제임.

# 01. RAG 도구 정의
# 먼저 앞에서 설명한 config 를 RAGTool 에 주입해 rag_tool 인스턴스를 생성함.
# 이어서 KB 주택시장 리뷰 PDF와 KB 연구보고서 웹페이지를 add() 메소드를 통해 벡터 데이터베이스에 추가함.
# 이 단계 까지가 지식 데이터 베이스 구축에 해당함.

# 1) RAG 도구 정의.
rag_tool = RagTool(
    name= "MyDocsRAG",
    description = "KB리포트를 기반으로 주택시장 질의에 답하는 RAG 툴",
    summarize= True,
    result_as_answer = True,
    verbose = True,
    config = config
)

# KB_주택시장 리뷰 PDF 추가.
rag_tool.add(
    data_type = "file",
    path="./knowledge/KB주택시장리뷰_2026년 1월호.pdf"
)

# KB_연구보고서 리스트 페이지 추가.
rag_tool.add(
    data_type="website",
    path="https://www.kbfg.com/kbresearch/report/reportList.do"
)

In [ ]:
# 02. Agent 정의
# housing_research_agent 는 실제로 Rag 툴을 호출해 문서를 검색하는 역할을 담당.
# tools = [rag_tool] 로 설정했기 때문에 에이전트는 자연어 프롬프트 안에서 RAG 도구를 활용함.
# responder_agent 는 검색 결과를 바탕으로 최종 리포트를 생성하는 에이전트로, 사람에게 읽히기 좋은 형태로 내용을 정리하는데 집중함.

# 2) 에이전트 정의
housing_research_agent = Agent(
    role="주택시장 리서치 에이전트",
    goal= (
        "KB 주택 시장 리뷰 및 관련 자료를 기반으로 한국 주택 시장 동향과 리스크를 정리한다."
    ),
    backstory = (
        "KB 금융지주 리서치센터에서 근무해 온 시니어 애널리스트로, 보고서와 데이터에 근거해 인사이트를 제공한다."
    ),
    tools = [rag_tool],
    verbose = True,
)

responder_agent = Agent(
    role="리포트 작성 에이전트",
    goal="검색된 내용을 바탕으로 이해하기 쉬운 한국어 분석 리포트를 작성한다.",
    backstory="경제 리포트와 시장 분석 문서를 다수 작성해 온 애널리스트이다.",
    verbose = True
)

In [ ]:
# 03. 태스크 정의
# research_task는 먼저 문서를 찾아 핵심 문장을 정리하는 단계임. expected_output 을 통해 어떤 형식으로 요약 노트를 남겨야 하는지 명확하게 지정함으로써 후속 태스크가 이를 그대로 재사용할 수 있도록 함.
# housing_analysis_task 는 context=[research_task]를 통해 앞선 태스크 결과를 입력으로 받어 최종 분석 리포트를 작성함.
# 이 테스트의 expected_output 를 리포트 탬플릿처럼 설계해 두면 항상 동일한 구조로 결과를 얻을 수 있음.

# 3) 태스크 정의
research_task = Task(
    description= (
        "다음 질문에 대해 KB 주택 시장 리뷰 및 연결 자료를 먼저 탐색하라.\n"
        "질문 : {question}\n\n"
        "요구사항 : \n"
        "1) 반드시 RAG 툴(MyDocsRAG)을 사용해 관련 문서를 검색한다.\n"
        "2) 후속 태스크가 이해하기 쉽도록 핵심 내용을 정리해 둔다.\n"
    ),
    expected_output = (
        "다음 정보를 포함하는 한국어 요약 노트를 작성한다.\n\n"
        "1. 관련성 높은 문단/페이지에서 인용한 핵심 문장 정리(3~7개 불릿)\n"
        "2. 각 인용 내용의 출처 표기(예: 'KB 주택시장 리뷰 2026년 1월호, p.12')\n"
    ),
    agent = housing_research_agent,
    tools = [rag_tool],
    verbose = True
)

housing_analysis_task = Task(
    description = (
        "다음 질문에 대해 KB 주택시장 리뷰 및 연결된 자료를 우선적으로 활용해 답변하라. \n"
        "질문 : {question} \n\n"
        "1) 보고서에 명시된 내용과 수치를 중심으로 정리한다. \n"
        "2) 인용한 내용이 어느 시기(연도/월)의 리뷰인지 간단히 언급한다. \n"
        "3) 문서에서 찾기 어려운 내용은 '해당 문서에서 명확히 찾기 어렵다' 고 표시한다.\n"
    ),
    expected_output = (
        "다음 구조의 한국어 분석 리포트를 작성한다. \n\n"
        "1. 한줄요약 \n"
        "2. 핵심 내용 정리 \n"
        "3. 세부 설명 (문단 형식) \n"
        "4. 출처, 참고 (예: 'KB 주택시장 리뷰 2026년 1월호 기준')"
    ),
    agent = responder_agent,
    context = [research_task],
    verbose = True
)

In [ ]:
# 04. 크루 구성 및 실행.
# 마지막으로 두 에이전트와 두개 태스크를 Crew 로 묶고 process = Process.Sequential 로 설정해 research_task -> housing_analysis_task 순서로 실행하게 함.
# 4) Crew 구성.
housing_crew = Crew(
    agents = [housing_research_agent, responder_agent],
    tasks = [research_task, housing_analysis_task],
    process = Process.sequential,
    verbose = True
)

# 05. 실행.
# kickoff() 호출 시 inputs = {"question": ...}을 전달하면 두 태스크와 description에서 {question} 자리로 같은 질문이 주입됨.
# 실행이 끝나면 result.raw에는 크루 전체 관점에서의 통합 출력이 각 태스크의 output.raw 에 단계별 상세 결과가 담기게 됨.

# 5) 실행 예시
if __name__ == "__main__":
    question = "2026년 1월 기준 서울 및 수도권 아파트 시장의 주요 특징과 리스크는 무엇인가?"

    result = housing_crew.kickoff(inputs = {"question": question})

    print("\n=== Crew 통합 결과 (result.raw) ===")
    print(result.raw)

    print("\n=== Research Task 출력 ===")
    print(research_task.output.raw)

    print("\n=== 분석 테스크(housing_analysis_task) 원문 출력 ===")
    print(housing_analysis_task.output.raw) 